In [0]:
GOLD_CONFIG = {

    "dim_products": {

        "source": "emain_databricks.silver.products",

        "target": "emain_databricks.gold.dim_products",

        "delta_path": "abfss://gold@emedallionstore01.dfs.core.windows.net/delta/dim_products",

        "columns": [
            ("id","product_id"),
            ("title","title"),
            ("category","category"),
            ("brand","brand"),
            ("price","price"),
            ("rating","rating"),
            ("stock","stock"),
            ("availabilityStatus","availability_status"),
            ("minimumOrderQuantity","minimum_order_quantity"),
            ("shippingInformation","shipping_information"),
            ("warrantyInformation","warranty_information")
        ],

        "primary_key":"product_id"

    },

    "dim_users": {

        "source":"emain_databricks.silver.users",

        "target":"emain_databricks.gold.dim_users",

        "delta_path":"abfss://gold@emedallionstore01.dfs.core.windows.net/delta/dim_users",

        "columns":[
            ("id","user_id"),
            ("firstName","first_name"),
            ("lastName","last_name"),
            ("email","email"),
            ("gender","gender"),
            ("age","age"),
            ("phone","phone"),
            ("birthDate","birth_date"),
            ("bloodGroup","blood_group"),
            ("eyeColor","eye_color"),
            ("company","company"),
            ("role","role")
        ],

        "primary_key":"user_id"

    }

}

In [0]:
products_df=spark.table("emain_databricks.silver.products")

In [0]:
from pyspark.sql.functions import col
def processing_table(dataset):
    config=GOLD_CONFIG[dataset]
    print("="*60)
    print(f"Processing {dataset.upper()}")
    print("="*60)
    df=spark.table(config["source"])
    columns=[]
    for source_col,target_col in config["columns"]:

        columns.append(
            col(source_col).alias(target_col)
        )

    gold_df=(
        df.select(*columns).dropDuplicates([config["primary_key"]])
    )
    (
        gold_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(config["delta_path"])
    )
    spark.sql(f"""DROP TABLE IF EXISTS {config["target"]}""")
    spark.sql(f"""create table if not exists {config["target"]} using delta location '{config["delta_path"]}' """)
    print(f"Rows Written : {gold_df.count()}")
    print(f"{dataset.upper()} Completed\n")

In [0]:
for dataset in GOLD_CONFIG:

    processing_table(dataset)

print("="*60)
print("DIMENSION TABLES COMPLETED")
print("="*60)

Processing DIM_PRODUCTS
Rows Written : 194
DIM_PRODUCTS Completed

Processing DIM_USERS
Rows Written : 208
DIM_USERS Completed

DIMENSION TABLES COMPLETED


In [0]:
spark.read.format("delta") \
    .load("abfss://silver@emedallionstore01.dfs.core.windows.net/delta/carts") \
    .printSchema()

root
 |-- discountedTotal: double (nullable = true)
 |-- id: integer (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- products: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- discountPercentage: double (nullable = true)
 |    |    |-- discountedTotal: double (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- price: double (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |    |    |-- thumbnail: string (nullable = true)
 |    |    |-- title: string (nullable = true)
 |    |    |-- total: double (nullable = true)
 |-- total: double (nullable = true)
 |-- totalProducts: integer (nullable = true)
 |-- totalQuantity: integer (nullable = true)
 |-- userId: integer (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- processing_timestamp: timestamp (nullable = true)
 |-- pipeline_name: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, explode

carts_df = spark.table("emain_databricks.silver.carts")
products_df = spark.table("emain_databricks.silver.products")
users_df = spark.table("emain_databricks.silver.users")

fact_df=(
    carts_df.withColumn("product",explode("products"))
)
fact_df.printSchema()
fact_df = (
    fact_df.select(
        col("id").alias("cart_id"),
        col("userId").alias("user_id"),
        col("product.id").alias("product_id"),
        col("product.title").alias("product_name"),
        col("product.price").alias("unit_price"),
        col("product.quantity").alias("quantity"),
        col("product.total").alias("line_total"),
        col("product.discountPercentage").alias("discount_percentage"),
        col("product.discountedTotal").alias("discounted_total"),
        col("processing_timestamp")
    )
)

root
 |-- discountedTotal: double (nullable = true)
 |-- id: integer (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- products: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- discountPercentage: double (nullable = true)
 |    |    |-- discountedTotal: double (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- price: double (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |    |    |-- thumbnail: string (nullable = true)
 |    |    |-- title: string (nullable = true)
 |    |    |-- total: double (nullable = true)
 |-- total: double (nullable = true)
 |-- totalProducts: integer (nullable = true)
 |-- totalQuantity: integer (nullable = true)
 |-- userId: integer (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- processing_timestamp: timestamp (nullable = true)
 |-- pipeline_name: string (nullable = true)
 |-- product: struct (nullable = true)
 |    |-- discountPercen

In [0]:
fact_df = (
    fact_df.join(
        products_df.select(
            col("id").alias("product_id"),
            "category",
            "brand",
            "rating",
            "stock",
            "price",
            "availabilityStatus"
        ),
        on="product_id",
        how="left"
    )
)
fact_df = (
    fact_df.join(
        users_df.select(
            col("id").alias("user_id"),
            "firstName",
            "lastName",
            "age",
            "gender",
            "email",
            "phone",
            "birthDate"
        ),
        on="user_id",
        how="left"
    )
)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

fact_df = (
    fact_df
    .withColumn(
        "gold_processing_timestamp",
        current_timestamp()
    )
    .withColumn(
        "pipeline_name",
        lit("gold_fact_pipeline")
    )
)
gold_path = "abfss://gold@emedallionstore01.dfs.core.windows.net/delta/fact_cart_items"

(
    fact_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_path)
)
spark.sql("""
DROP TABLE IF EXISTS emain_databricks.gold.fact_cart_items
""")

spark.sql(f"""
CREATE TABLE emain_databricks.gold.fact_cart_items
USING DELTA
LOCATION '{gold_path}'
""")

DataFrame[]

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM emain_databricks.gold.fact_cart_items
""").show()

+--------+
|COUNT(*)|
+--------+
|     756|
+--------+



In [0]:
from pyspark.sql.functions import col,sum,avg,count,countDistinct,when

fact_df=spark.table("emain_databricks.gold.fact_cart_items")

product_summary=(
    fact_df
    .groupBy(
        "product_id",
        "product_name",
        "category",
        "brand"
    )
    .agg(
        sum("quantity").alias("total_quantity_sold"),
        sum("line_total").alias("total_revenue"),
        avg("unit_price").alias("average_selling_price"),
        count("cart_id").alias("number_of_orders")
    )
)

product_path="abfss://gold@emedallionstore01.dfs.core.windows.net/delta/product_sales_summary"

(
    product_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .save(product_path)
)

spark.sql("DROP TABLE IF EXISTS emain_databricks.gold.product_sales_summary")

spark.sql(f"""
CREATE TABLE emain_databricks.gold.product_sales_summary
USING DELTA
LOCATION '{product_path}'
""")

category_summary=(
    fact_df
    .groupBy("category")
    .agg(
        sum("line_total").alias("category_revenue"),
        sum("quantity").alias("quantity_sold"),
        countDistinct("cart_id").alias("orders")
    )
)

category_path="abfss://gold@emedallionstore01.dfs.core.windows.net/delta/category_sales_summary"

(
    category_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .save(category_path)
)

spark.sql("DROP TABLE IF EXISTS emain_databricks.gold.category_sales_summary")

spark.sql(f"""
CREATE TABLE emain_databricks.gold.category_sales_summary
USING DELTA
LOCATION '{category_path}'
""")

customer_summary=(
    fact_df
    .groupBy(
        "user_id",
        "firstName",
        "lastName"
    )
    .agg(
        countDistinct("cart_id").alias("total_orders"),
        sum("line_total").alias("total_spend"),
        avg("line_total").alias("average_order_value"),
        sum("quantity").alias("items_purchased")
    )
)

customer_path="abfss://gold@emedallionstore01.dfs.core.windows.net/delta/customer_summary"

(
    customer_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .save(customer_path)
)

spark.sql("DROP TABLE IF EXISTS emain_databricks.gold.customer_summary")

spark.sql(f"""
CREATE TABLE emain_databricks.gold.customer_summary
USING DELTA
LOCATION '{customer_path}'
""")

inventory_summary=(
    spark.table("emain_databricks.gold.dim_products")
    .withColumn(
        "inventory_status",
        when(col("stock")<=0,"Out of Stock")
        .when(col("stock")<=10,"Low Stock")
        .otherwise("In Stock")
    )
)

inventory_path="abfss://gold@emedallionstore01.dfs.core.windows.net/delta/inventory_summary"

(
    inventory_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .save(inventory_path)
)

spark.sql("DROP TABLE IF EXISTS emain_databricks.gold.inventory_summary")

spark.sql(f"""
CREATE TABLE emain_databricks.gold.inventory_summary
USING DELTA
LOCATION '{inventory_path}'
""")

display(product_summary)
display(category_summary)
display(customer_summary)
display(inventory_summary)

product_id,product_name,category,brand,total_quantity_sold,total_revenue,average_selling_price,number_of_orders
162,Blue Frock,tops,null,14,419.86,29.99,4
113,Generic Motorcycle,motorcycle,Generic Motors,28,111999.72,3999.989999999999,7
122,iPhone 6,smartphones,Apple,17,5099.83,299.99,6
138,Baseball Ball,sports-accessories,null,9,80.91,8.99,4
86,Man Short Sleeve Shirt,mens-shirts,Casual Comfort,13,259.87,19.99,3
104,Apple iPhone Charger,mobile-accessories,Apple,7,139.93,19.99,3
24,Fish Steak,groceries,null,20,299.79999999999995,14.99,5
123,iPhone 13 Pro,smartphones,Apple,13,14299.869999999999,1099.99,4
129,Realme X,smartphones,Realme,20,5999.8,299.99,8
8,Dior J'adore,fragrances,Dior,19,1709.81,89.99,8


category,category_revenue,quantity_sold,orders
tops,2409.28,72,22
motorcycle,669999.1599999998,84,22
smartphones,82348.23000000003,177,52
sports-accessories,4086.389999999999,161,48
mens-shirts,1512.4400000000003,56,17
mobile-accessories,27586.15,185,55
groceries,1985.3800000000017,342,91
fragrances,5649.359999999999,64,22
sunglasses,1497.4600000000003,54,16
mens-shoes,8589.249999999998,75,22


user_id,firstName,lastName,total_orders,total_spend,average_order_value,items_purchased
1,Emily,Johnson,1,13037.879999999997,3259.4699999999993,12
2,Michael,Williams,1,139.92999999999998,69.96499999999999,7
3,Sophia,Brown,1,1794.8500000000001,299.1416666666667,15
4,James,Davis,1,689.93,344.965,7
5,Emma,Miller,1,1467.88,489.29333333333335,12
6,Olivia,Wilson,1,55453.85999999999,13863.464999999998,14
7,Alexander,Jones,1,1229.8600000000001,204.9766666666667,14
8,Ava,Taylor,1,6337.83,1056.305,17
9,Ethan,Martinez,1,29709.9,7427.475,10
10,Isabella,Anderson,1,152.44,76.22,6


product_id,title,category,brand,price,rating,stock,availability_status,minimum_order_quantity,shipping_information,warranty_information,inventory_status
12,Annibale Colombo Sofa,furniture,Annibale Colombo,2599.99,3.92,58,In Stock,1,Ships in 1 week,Lifetime warranty,In Stock
18,Cat Food,groceries,null,8.99,3.13,55,In Stock,18,Ships overnight,1 year warranty,In Stock
38,Rice,groceries,null,5.99,3.18,59,In Stock,20,Ships overnight,6 months warranty,In Stock
67,Mug Tree Stand,kitchen-accessories,null,15.99,2.64,88,In Stock,45,Ships in 1 month,1 year warranty,In Stock
70,Red Tongs,kitchen-accessories,null,6.99,4.42,82,In Stock,23,Ships in 1 week,No warranty,In Stock
93,Brown Leather Belt Watch,mens-watches,Fashion Timepieces,89.99,4.19,52,In Stock,7,Ships in 1 month,1 year warranty,In Stock
161,Samsung Galaxy Tab White,tablets,Samsung,349.99,3.72,104,In Stock,5,Ships overnight,3 months warranty,In Stock
186,Calvin Klein Heel Shoes,womens-shoes,Calvin Klein,79.99,4.92,93,In Stock,6,Ships in 1 week,2 year warranty,In Stock
190,IWC Ingenieur Automatic Steel,womens-watches,IWC,4999.99,2.93,90,In Stock,1,Ships overnight,1 year warranty,In Stock
16,Apple,groceries,null,1.99,4.19,8,In Stock,7,Ships in 2 weeks,3 year warranty,Low Stock
